# Ethical Financial Data Analytics
Single-cell Google Colab demonstration for Case Study 41. Uses synthetic transaction data only.

**To execute:** run the cell, change `CONSENT_GRANTED = False` to `True`, and rerun it. No local installation is required.

In [ ]:
# Ethical Financial Data Analytics — single-cell Colab execution
# Educational prototype using synthetic transactions only. Do not use real financial data.

import hashlib
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

SEED = 41
rng = np.random.default_rng(SEED)
audit_log = []

def audit(event, detail):
    audit_log.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "event": event,
        "detail": detail
    })

display(Markdown("# Ethical Financial Data Analytics\nSynthetic-data responsible-AI demonstration"))

# CONSENT: in Colab, explicitly set this to True to proceed.
CONSENT_GRANTED = False
if not CONSENT_GRANTED:
    display(Markdown("## Consent required\nSet CONSENT_GRANTED = True near the top of this cell and rerun to enable analysis."))
    audit("consent_checked", "Consent not granted; analysis stopped")
else:
    audit("consent_granted", "User opted in for this demo session")

    # SYNTHETIC DATA: no real customer records are loaded.
    n_customers = 240
    categories = ["Housing", "Groceries", "Transport", "Dining", "Shopping", "Utilities", "Health", "Entertainment"]
    weights = np.array([.24, .18, .10, .10, .11, .09, .08, .10])
    records = []
    for i in range(1, n_customers + 1):
        income = float(np.clip(rng.normal(52000, 19000), 16000, 140000))
        age = int(rng.integers(18, 70))
        age_group = "18–29" if age < 30 else ("30–49" if age < 50 else "50+")
        income_band = "Lower" if income < 35000 else ("Middle" if income < 70000 else "Higher")
        cid = f"C{i:04d}"
        base_monthly_spend = income * float(rng.uniform(.48, .88))
        category_weights = rng.dirichlet(weights * 35)
        for month in range(1, 7):
            month_spend = base_monthly_spend * float(rng.normal(1, .09))
            amounts = rng.multinomial(max(1, int(month_spend)), category_weights)
            for category, amount in zip(categories, amounts):
                if amount > 0:
                    records.append({
                        "customer_id": cid,
                        "age_group": age_group,
                        "income_band": income_band,
                        "monthly_income_inr": round(income, 2),
                        "month": f"2026-{month:02d}",
                        "category": category,
                        "amount_inr": float(amount)
                    })
    transactions = pd.DataFrame(records)
    audit("synthetic_data_created", f"{len(transactions)} synthetic transaction rows")

    # DATA MINIMIZATION: analyze only the fields needed for insights.
    minimal = transactions[["customer_id", "month", "category", "amount_inr"]].copy()
    category_summary = minimal.groupby("category", as_index=False)["amount_inr"].sum().sort_values("amount_inr", ascending=False)
    monthly_summary = minimal.groupby("month", as_index=False)["amount_inr"].sum()
    audit("data_minimized", "Analysis restricted to customer ID, month, category and amount")

    # PRIVACY: hashed IDs are pseudonyms, not guaranteed anonymization.
    minimal["masked_customer_id"] = minimal["customer_id"].map(
        lambda x: "ID-" + hashlib.sha256(x.encode()).hexdigest()[:8]
    )
    safe_preview = minimal[["masked_customer_id", "month", "category", "amount_inr"]].head(15)

    # TRANSPARENT RULE-BASED PROMPTS, not financial advice.
    discretionary = transactions[transactions["category"].isin(["Dining", "Shopping", "Entertainment"])]
    discretionary_spend = discretionary.groupby("customer_id")["amount_inr"].sum()
    median_spend = float(discretionary_spend.median())
    prompts = pd.DataFrame({
        "customer_id": discretionary_spend.index,
        "discretionary_spend_inr": discretionary_spend.values
    })
    prompts["suggested_prompt"] = np.where(
        prompts["discretionary_spend_inr"] > median_spend,
        "Review discretionary spending and consider a realistic savings target.",
        "Consider a small savings goal and review recurring bills."
    )
    audit("recommendations_generated", "Transparent median-threshold rule applied to all customers")

    # FAIRNESS AUDIT: demographic fields are for auditing, not recommendation inputs.
    customer_groups = transactions[["customer_id", "age_group", "income_band"]].drop_duplicates()
    fairness = prompts.merge(customer_groups, on="customer_id", how="left")
    fairness["received_review_prompt"] = fairness["discretionary_spend_inr"] > median_spend
    fairness_rows = []
    for dimension in ["age_group", "income_band"]:
        for group, part in fairness.groupby(dimension):
            fairness_rows.append({
                "audit_dimension": dimension,
                "group": group,
                "customers": int(len(part)),
                "review_prompt_rate_pct": round(float(part["received_review_prompt"].mean() * 100), 1)
            })
    fairness_summary = pd.DataFrame(fairness_rows)
    audit("fairness_audit_run", "Compared prompt rates by synthetic age and income groups")

    # DISPLAY RESULTS
    display(Markdown("## Dataset overview"))
    display(pd.DataFrame([{
        "Synthetic customers": int(transactions["customer_id"].nunique()),
        "Transaction rows": int(len(transactions)),
        "Total six-month spend (INR)": round(float(transactions["amount_inr"].sum()), 2),
        "Average monthly income (INR)": round(float(transactions.drop_duplicates("customer_id")["monthly_income_inr"].mean()), 2)
    }]))
    display(Markdown("## Aggregate spending by category"))
    display(category_summary)
    display(Markdown("## Monthly spending trend"))
    display(monthly_summary)
    display(Markdown("## Masked transaction preview"))
    display(safe_preview)
    display(Markdown("## Example personalized prompts"))
    display(prompts.head(10))
    display(Markdown("## Fairness monitoring (screening only)"))
    display(fairness_summary)
    display(Markdown("## Governance checklist"))
    governance = pd.DataFrame([
        ["Consent and purpose limitation", "Opt-in gate before analysis", "Demonstrated"],
        ["Data minimization", "Restricted analysis columns", "Demonstrated"],
        ["Privacy", "Synthetic data and hashed display IDs", "Illustrative; hashing is not anonymization"],
        ["Bias and fairness", "Group-level prompt-rate comparison", "Screening only; not proof of fairness"],
        ["Security", "No real data or secrets used", "Production security not implemented"],
        ["Accountability", "Session audit log", "Temporary, not tamper-proof"]
    ], columns=["Control", "Mechanism", "Status"])
    display(governance)
    display(Markdown("## Audit log"))
    display(pd.DataFrame(audit_log))
    display(Markdown("**Limitations:** Educational proof of concept only, not a production banking system, compliance certification, or financial advice. Fairness rates need contextual review; differences alone do not prove discrimination."))
